In [7]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [8]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

display(run(load_queries("sql/01_scope.sql"), "scope_reconcile"))

,raw_rows,scoped_rows,rows_removed_by_scope
0,20405666,20405666,0


In [9]:
integ = load_queries("sql/01_audit_integrity.sql")
con.execute(integ["create_trips_hashed"])

for name in ["zones_key_check", "zones_special_rows", "join_reconcile", "borough_coverage",
             "unknown_zone_trips", "flag_values", "flag_consistency", "cbd_fee_timing"]:
    print(f"\n=== {name} ===")
    display(run(integ, name))


=== zones_key_check ===


,n_rows,n_distinct_ids,min_id,max_id,n_rows_with_true_null
0,265,265,1,265,0



=== zones_special_rows ===


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,264,Unknown,N/A,N/A
2,265,N/A,Outside of NYC,N/A



=== join_reconcile ===


,scoped_rows,joined_rows,pickup_unmatched,dropoff_unmatched
0,20405666,20405666,0,0


... (6 outputs left out)


=== flag_consistency ===


,hvfhs_license_num,n_rows,shared_match_without_request,wav_requested_not_matched,access_a_ride_y,access_a_ride_and_wav_request
0,HV0003,15356455,0,0,14491,1706
1,HV0005,5049211,424,0,5979,636



=== cbd_fee_timing ===


,period,n_rows,n_fee_positive,pct_fee_positive
0,1_before_jan5,2477942,1184,0.048
1,2_jan5_onward,17927724,6299028,35.136


In [10]:
for name in ["duplicate_summary", "duplicate_groups", "duplicate_by_company"]:
    print(f"\n=== {name} ===")
    display(run(integ, name))


=== duplicate_summary ===


,n_rows,n_distinct_exact,exact_duplicate_excess_rows,n_distinct_fingerprint,fingerprint_duplicate_excess_rows
0,20405666,20405666,0,20405666,0



=== duplicate_groups ===


,group_size,n_exact_variants,n_groups,n_rows_in_groups



=== duplicate_by_company ===


,company,n_groups,n_rows_in_groups,excess_rows,n_valid_wait_rows


In [11]:
s = run(integ, "column_summary")
display(s[["column_name", "column_type", "min", "max", "approx_unique", "null_percentage"]])

,column_name,column_type,min,max,approx_unique,null_percentage
0,hvfhs_license_num,VARCHAR,HV0003,HV0005,1,0.00
1,dispatching_base_num,VARCHAR,B03404,B03406,2,0.00
2,originating_base_num,VARCHAR,B00887,B03406,6,24.68
3,request_datetime,TIMESTAMP,2024-12-31 22:31:09,2025-02-01 00:10:00,2959087,0.00
4,on_scene_datetime,TIMESTAMP,2024-12-31 23:25:35,2025-01-31 23:59:59,2914548,24.68
5,pickup_datetime,TIMESTAMP,2025-01-01 00:00:00,2025-01-31 23:59:59,2959087,0.00
6,dropoff_datetime,TIMESTAMP,2025-01-01 00:01:48,2025-02-01 01:50:34,3005009,0.00
7,PULocationID,INTEGER,1,265,298,0.00
8,DOLocationID,INTEGER,1,265,298,0.00
9,trip_miles,DOUBLE,0.0,507.22,53505,0.00


In [13]:
fu = load_queries("sql/01_audit_integrity_followups.sql")
for name in fu:
    print(f"\n=== {name} ===")
    display(run(fu, name))


=== lyft_shared_exception_flags ===


,wav_request_flag,wav_match_flag,access_a_ride_flag,n_rows
0,N,N,N,370
1,N,Y,N,52
2,N,N,Y,2



=== lyft_shared_exception_by_base ===


,dispatching_base_num,originating_base_num,n_exception,n_lyft_rows,pct_exception
0,B03406,None,424,5035135,0.0084
1,B03406,B03406,0,14076,0.0000



=== lyft_shared_exception_by_day ===


,pickup_date,n_exception,n_lyft_rows,pct_exception
0,2025-01-01,12,157741,0.0076
1,2025-01-02,1,124595,0.0008
2,2025-01-03,1,150350,0.0007
3,2025-01-04,5,168086,0.0030
4,2025-01-05,0,143313,0.0000
5,2025-01-06,4,150266,0.0027
6,2025-01-07,6,165428,0.0036
7,2025-01-08,15,175763,0.0085
8,2025-01-09,7,186884,0.0037
9,2025-01-10,5,178267,0.0028


... (6 outputs left out)


=== cbd_fee_values ===


,period,hvfhs_license_num,cbd_congestion_fee,n_rows
0,1_before_jan5,HV0003,1.5,1184
1,2_jan5_onward,HV0003,1.5,4637147
2,2_jan5_onward,HV0005,1.5,1661876
3,2_jan5_onward,HV0005,3.0,5



=== cbd_prejan5_zones ===


,side,Borough,Zone,n_rows
0,pickup,Queens,JFK Airport,100
1,pickup,Manhattan,East Village,58
2,pickup,Manhattan,Times Sq/Theatre District,47
3,pickup,Manhattan,Lower East Side,45
4,pickup,Manhattan,Clinton East,45
5,pickup,Manhattan,East Chelsea,37
6,pickup,Manhattan,Greenwich Village South,36
7,pickup,Manhattan,West Village,36
8,dropoff,N/A,Outside of NYC,191
9,dropoff,Brooklyn,Stuyvesant Heights,29
